## HIL-004 Data Inspection and Cleaning

This notebook documents a conservative cleaning workflow for the HIL-004 comprehensive plan snapshot. The raw ArcGIS response is preserved unchanged; derived cleaned attributes, geometry, and quality flags are exported separately.

- Source: City of Hillsboro GIS
- Dataset: Comprehensive plan
- Snapshot: `2026-08-26`
- Geometry: Polygon
- Coordinate system: Web Mercator Auxiliary Sphere (`WKID 102100`, latest `WKID 3857`)


In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_VERSION = "2026-08-26"
DATA_ROOT = PROJECT_ROOT / DATA_VERSION
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
RAW_FILE = RAW_DIR / "HIL-004.json"
PROCESSED_FILE = PROCESSED_DIR / "HIL-004_cleaned.json"
MANIFEST_FILE = PROCESSED_DIR / "HIL-004_cleaning_manifest.json"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

df = pd.DataFrame([feature["attributes"] for feature in raw_data["features"]])
print(f"Loaded {RAW_FILE.name}: {len(df):,} rows, {len(df.columns):,} source fields")
print(f"Geometry type: {raw_data.get('geometryType')}")

Loaded HIL-004.json: 297 rows, 9 source fields
Geometry type: esriGeometryPolygon


## Raw Structure and Missingness

The audit checks sparse tracking metadata, comprehensive-plan codes, identifier uniqueness, and polygon geometry before any transformation.

In [2]:
import numpy as np

schema_df = pd.DataFrame(raw_data["fields"])[["name", "alias", "type"]]
missing = df.isna().sum().to_frame("missing_count")
missing["missing_percent"] = (missing["missing_count"] / len(df) * 100).round(1)
identifier_audit = pd.DataFrame({
    "field": ["OBJECTID", "GlobalID"],
    "unique_values": [df["OBJECTID"].nunique(), df["GlobalID"].nunique()],
    "missing_values": [df["OBJECTID"].isna().sum(), df["GlobalID"].isna().sum()],
    "duplicate_values": [df["OBJECTID"].duplicated().sum(), df["GlobalID"].duplicated().sum()],
})

print("Field types:")
display(schema_df)
print("Missingness:")
display(missing.sort_values("missing_count", ascending=False))
print("Identifier audit:")
display(identifier_audit)
print("Comprehensive-plan descriptions:")
display(df["COMP_DESCR"].value_counts(dropna=False).to_frame("record_count"))


Field types:


,name,alias,type
0,GlobalID,GlobalID,esriFieldTypeGlobalID
1,Tracking_CreateID,Create ID,esriFieldTypeString
2,UTC_CreateDate,Create Date,esriFieldTypeDate
3,Tracking_EditID,Edit ID,esriFieldTypeString
4,UTC_EditDate,Edit Date,esriFieldTypeDate
5,OBJECTID,OBJECTID,esriFieldTypeOID
6,COMP_DESCR,COMP_DESCR,esriFieldTypeString
7,Shape__Area,SHAPE.STArea(),esriFieldTypeDouble
8,Shape__Length,SHAPE.STLength(),esriFieldTypeDouble


Missingness:


,missing_count,missing_percent
Tracking_CreateID,260,87.5
UTC_CreateDate,260,87.5
Tracking_EditID,244,82.2
UTC_EditDate,244,82.2
GlobalID,0,0.0
OBJECTID,0,0.0
COMP_DESCR,0,0.0
Shape__Area,0,0.0
Shape__Length,0,0.0


Identifier audit:


,field,unique_values,missing_values,duplicate_values
0,OBJECTID,297,0,0
1,GlobalID,297,0,0


Comprehensive-plan descriptions:


,record_count
COMP_DESCR,
RM,75
OS,39
PF,37
RH,30
RL,28
C,21
IN,16
FP,12
MU,11


## Geometry and Derived Clean View

Polygon geometry is converted for validation only. Invalid polygons are repaired in the derived view with `make_valid`; source rings and source attributes remain unchanged.

In [3]:
from shapely.geometry import Polygon


def arcgis_polygon_to_shapely(geometry):
    rings = geometry.get("rings", []) if geometry else []
    if not rings:
        return None
    try:
        return Polygon(rings[0], holes=rings[1:])
    except (TypeError, ValueError):
        return None

geometry_records = []
for feature in raw_data["features"]:
    geometry = feature.get("geometry") or {}
    rings = geometry.get("rings", [])
    polygon = arcgis_polygon_to_shapely(geometry)
    geometry_records.append({
        "geometry": polygon,
        "ring_count": len(rings),
        "vertex_count": sum(len(ring) for ring in rings),
    })

geometry_df = pd.DataFrame(geometry_records)
geometry_df["valid_source_geometry"] = geometry_df["geometry"].map(
    lambda geometry: geometry is not None and geometry.is_valid
)
geometry_df["empty_geometry_flag"] = geometry_df["geometry"].isna()
geometry_df["area"] = geometry_df["geometry"].map(
    lambda geometry: geometry.area if geometry is not None else np.nan
)
geometry_df["length"] = geometry_df["geometry"].map(
    lambda geometry: geometry.length if geometry is not None else np.nan
)

print("Geometry audit:")
display(geometry_df.drop(columns="geometry").describe(include="all"))
print("Invalid geometries:", int((~geometry_df["valid_source_geometry"] & ~geometry_df["empty_geometry_flag"]).sum()))
print("Empty geometries:", int(geometry_df["empty_geometry_flag"].sum()))


Geometry audit:


,ring_count,vertex_count,valid_source_geometry,empty_geometry_flag,area,length
count,297.000000,297.00000,297,297,2.970000e+02,297.000000
unique,NaN,NaN,2,1,NaN,NaN
top,NaN,NaN,True,False,NaN,NaN
freq,NaN,NaN,293,297,NaN,NaN
mean,1.094276,184.69697,NaN,NaN,5.281071e+05,3114.762603
std,0.624284,583.34695,NaN,NaN,2.650678e+06,7249.011022
min,1.000000,4.00000,NaN,NaN,3.686841e-03,0.418323
25%,1.000000,24.00000,NaN,NaN,3.644773e+04,900.407639
50%,1.000000,52.00000,NaN,NaN,8.324296e+04,1439.230831
75%,1.000000,129.00000,NaN,NaN,2.365125e+05,2647.031710


Invalid geometries: 4
Empty geometries: 0


In [4]:
from shapely.validation import make_valid

analysis_df = df.copy()
text_columns = analysis_df.select_dtypes(include=["str"]).columns
for column in text_columns:
    analysis_df[column] = analysis_df[column].map(
        lambda value: None if isinstance(value, str) and not value.strip() else value
    )

for column in ["UTC_CreateDate", "UTC_EditDate"]:
    analysis_df[column] = pd.to_datetime(
        analysis_df[column], unit="ms", utc=True, errors="coerce"
    ).dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    analysis_df[column] = analysis_df[column].where(analysis_df[column].notna(), None)

analysis_df["GEOMETRY_REPAIRED"] = ~geometry_df["valid_source_geometry"] & ~geometry_df["empty_geometry_flag"]
analysis_df["GEOMETRY_EMPTY_QA_FLAG"] = geometry_df["empty_geometry_flag"]
analysis_df["GEOMETRY_QA_FLAG"] = analysis_df["GEOMETRY_REPAIRED"]
analysis_df["OBJECTID_DUPLICATE_QA_FLAG"] = analysis_df["OBJECTID"].duplicated(keep=False)
analysis_df["GLOBALID_DUPLICATE_QA_FLAG"] = analysis_df["GlobalID"].duplicated(keep=False)
analysis_df["NEGATIVE_AREA_QA_FLAG"] = df["Shape__Area"] < 0

completely_missing_fields = [column for column in df.columns if df[column].isna().all()]
analysis_df_compact = analysis_df.drop(columns=completely_missing_fields)
analysis_geometry_by_id = {}
for index, row in geometry_df.iterrows():
    geometry = row["geometry"]
    analysis_geometry_by_id[df.iloc[index]["OBJECTID"]] = (
        make_valid(geometry) if geometry is not None and not geometry.is_valid else geometry
    )

print("Completely missing source fields:", completely_missing_fields)
print("Derived analytical fields:", len(analysis_df_compact.columns))
print("QA flag totals:")
display(analysis_df.filter(like="_QA_FLAG").sum().to_frame("flagged_records"))


Completely missing source fields: []
Derived analytical fields: 15
QA flag totals:


,flagged_records
GEOMETRY_EMPTY_QA_FLAG,0
GEOMETRY_QA_FLAG,4
OBJECTID_DUPLICATE_QA_FLAG,0
GLOBALID_DUPLICATE_QA_FLAG,0
NEGATIVE_AREA_QA_FLAG,0


## Validation and Export

Rows and source geometries are preserved one-for-one. Repaired polygons are written only to the processed derived view, while the original ArcGIS rings remain unchanged in the raw snapshot.

In [8]:
assert len(raw_data["features"]) == len(analysis_df) == 297
assert analysis_df["OBJECTID"].is_unique
assert analysis_df["GlobalID"].is_unique
assert not analysis_df["GEOMETRY_EMPTY_QA_FLAG"].any()
assert all(
    geometry is not None and geometry.is_valid and geometry.area >= 0
    for geometry in analysis_geometry_by_id.values()
)
assert analysis_df_compact.columns.is_unique

print("Cleaning validation passed")
print(f"Rows preserved: {len(analysis_df):,}")
print(f"Compact fields: {len(analysis_df_compact.columns):,}")
print(f"Repaired geometries: {int(analysis_df['GEOMETRY_REPAIRED'].sum())}")
print(f"Negative source areas flagged: {int(analysis_df['NEGATIVE_AREA_QA_FLAG'].sum())}")

Cleaning validation passed
Rows preserved: 297
Compact fields: 15
Repaired geometries: 4
Negative source areas flagged: 0


In [9]:
import math


def json_safe(value):
    if value is None or value is pd.NA:
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def geometry_to_arcgis_rings(geometry):
    if geometry is None:
        return None
    polygons = [geometry] if geometry.geom_type == "Polygon" else list(geometry.geoms)
    rings = []
    for polygon in polygons:
        rings.append([[json_safe(x), json_safe(y)] for x, y in polygon.exterior.coords])
        rings.extend(
            [[[json_safe(x), json_safe(y)] for x, y in interior.coords]
             for interior in polygon.interiors]
        )
    return {"rings": rings}

schema_records = [field.copy() for field in raw_data["fields"]]
source_field_names = {field["name"] for field in schema_records}
for field_name in analysis_df_compact.columns:
    if field_name not in source_field_names:
        schema_records.append({
            "name": field_name,
            "alias": field_name,
            "type": "esriFieldTypeSmallInteger",
        })

processed_features = []
for index, raw_feature in enumerate(raw_data["features"]):
    attributes = {
        field: json_safe(value)
        for field, value in analysis_df_compact.iloc[index].to_dict().items()
    }
    object_id = df.iloc[index]["OBJECTID"]
    processed_features.append({
        "attributes": attributes,
        "geometry": geometry_to_arcgis_rings(analysis_geometry_by_id[object_id]),
    })

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
processed_data = {
    "objectIdFieldName": raw_data.get("objectIdFieldName"),
    "globalIdFieldName": raw_data.get("globalIdFieldName"),
    "geometryType": raw_data.get("geometryType"),
    "spatialReference": raw_data.get("spatialReference"),
    "fields": schema_records,
    "features": processed_features,
    "cleaning_summary": {
        "source_file": RAW_FILE.name,
        "data_version": DATA_VERSION,
        "rows": len(processed_features),
        "compact_attribute_fields": len(analysis_df_compact.columns),
        "completely_missing_source_fields": completely_missing_fields,
        "repaired_geometries": int(analysis_df["GEOMETRY_REPAIRED"].sum()),
        "negative_source_areas": int(analysis_df["NEGATIVE_AREA_QA_FLAG"].sum()),
    },
}

with open(PROCESSED_FILE, "w", encoding="utf-8") as file:
    json.dump(processed_data, file, indent=2, ensure_ascii=True)

In [10]:
from datetime import datetime

cleaning_manifest = [
    {
        "field_or_scope": "Blank text values",
        "action": "Represent blank or whitespace-only strings as missing in derived attributes",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "UTC_CreateDate and UTC_EditDate",
        "action": "Convert ArcGIS epoch milliseconds to ISO-8601 UTC strings",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "Sparse tracking metadata",
        "action": "Preserve missing values without imputation",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "Invalid polygon geometry",
        "action": "Use make_valid in derived geometry view",
        "source_preserved": True,
        "review_flag": "GEOMETRY_QA_FLAG",
    },
    {
        "field_or_scope": "Duplicate identifiers and negative areas",
        "action": "Retain source records and flag for review",
        "source_preserved": True,
        "review_flag": "OBJECTID_DUPLICATE_QA_FLAG / GLOBALID_DUPLICATE_QA_FLAG / NEGATIVE_AREA_QA_FLAG",
    },
]

manifest_data = {
    "dataset": "HIL-004",
    "dataset_name": "comprehensive_plan",
    "data_version": DATA_VERSION,
    "created_utc": datetime.now().astimezone().isoformat(),
    "source_file": str(RAW_FILE),
    "processed_file": str(PROCESSED_FILE),
    "rows": len(processed_features),
    "cleaning_manifest": cleaning_manifest,
}

with open(MANIFEST_FILE, "w", encoding="utf-8") as file:
    json.dump(manifest_data, file, indent=2, ensure_ascii=True)

with open(PROCESSED_FILE, "r", encoding="utf-8") as file:
    reloaded_processed_data = json.load(file)
with open(MANIFEST_FILE, "r", encoding="utf-8") as file:
    reloaded_manifest_data = json.load(file)

assert len(reloaded_processed_data["features"]) == len(raw_data["features"])
assert len(reloaded_manifest_data["cleaning_manifest"]) == len(cleaning_manifest)
assert reloaded_processed_data["features"][0]["attributes"]["OBJECTID"] == int(df.iloc[0]["OBJECTID"])
print(f"Wrote: {PROCESSED_FILE}")
print(f"Wrote: {MANIFEST_FILE}")
print("Reload validation passed")


Wrote: c:\Users\John\Documents\hillsborogis\2026-08-26\processed\HIL-004_cleaned.json
Wrote: c:\Users\John\Documents\hillsborogis\2026-08-26\processed\HIL-004_cleaning_manifest.json
Reload validation passed
